# Final experiments: stain robustness on Camelyon17 (Group 19)

Runtime → Change runtime type → **T4 GPU**. Every run saves its own result file and is skipped if it already exists, so you can stop and resume at any time.

**What this notebook adds to the interim study**

| Part | What | Hospital | Runs |
|---|---|---|---|
| D1 | Check the size measurement against the official ITC / micro / macro slide labels | all (labels only) | none, 2 min |
| A1 | HED strength ablation, σ = 0.05, 0.1 and 0.4 (hed, both) | 1 | 18 |
| A2 | Normalisation reference ablation, one lab at a time (h0, h3, h4) | 1 | 9 |
| A3 | Stronger backbones: histology ResNet-18 (Ciga 2022), histology ResNet-50 (Kang 2023) | 1 | 12 + 12 |
| B | Score every patch of hospital 1, then the **one-time final test on hospital 2** | 1, 2 | inference only |
| C | Leave-one-hospital-out, all 5 hospitals | all | 60 |
| D2 | Metastasis size from each model's tumour calls | 1, 2 | analysis only |

**Settings are frozen now** (σ = 0.2, pooled reference, 5 epochs, 40,000 patches, threshold from id_val). The ablations are reported as sensitivity checks; they do not change the main settings. That is what makes it safe to open hospital 2 in part B.

**Suggested split between us** (each on our own Colab account, all writing to the same shared Drive folder):
- Person 1: A0 (main checkpoints), A3, then B and D2. Needs the main ResNet-18 checkpoints.
- Person 2: D1, A1, A2.
- Person 3: C (the longest part, 60 runs, no checkpoints needed).

## 0. Setup (everyone)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE   = "/content/drive/MyDrive/ai_project"      # shared project folder (add a shortcut to it in your own Drive)
RESULTS = f"{DRIVE}/results"; CKPT = f"{DRIVE}/ckpt"; CACHE = f"{DRIVE}/cache"
FINAL   = f"{DRIVE}/final";   SIZE = f"{DRIVE}/size"; SUMMARY = f"{DRIVE}/summary"; WEIGHTS = f"{DRIVE}/weights"
STAGE   = f"{DRIVE}/stage_labels.csv"
MAIN_TAG = "v3"        # tag of the interim ResNet-18 runs: check the file names in RESULTS
SEEDS = "0 1 2"
for d in (RESULTS, CKPT, CACHE, FINAL, SIZE, SUMMARY, WEIGHTS):
    os.makedirs(d, exist_ok=True)

In [ ]:
# Code: upload ai-project-final.zip to the Drive folder (or push it to GitHub and set USE_GIT = True)
USE_GIT = False
if USE_GIT:
    !git clone https://github.com/myrahhussain/ai-project.git /content/ai-project
else:
    !unzip -q -o "$DRIVE/ai-project-final.zip" -d /content
%cd /content/ai-project
!pip -q install wilds scikit-image opencv-python-headless pyyaml tqdm
!python -m pytest tests -q 2>&1 | tail -2

In [ ]:
# Data: about 10 GB, about 5 minutes. Needed in every new session.
from wilds import get_dataset
ds = get_dataset(dataset='camelyon17', download=True, root_dir='/content/data')
print(len(ds), "patches; metadata columns:", list(ds._metadata_df.columns))

In [ ]:
# How long will a run take on this GPU? (random weights, nothing downloaded)
!python scripts/speed_test.py --backbones resnet18 resnet18_histo resnet50_histo

## D1. Can we measure metastasis size from patches? (Person 2, no GPU needed, ~2 min)

CAMELYON17 labels every slide by the size of its largest metastasis: isolated tumour cells (≤ 0.2 mm), micro (0.2–2 mm), macro (> 2 mm). This step measures each of the 50 slides from the pathologists' tumour patches and checks the result against those official labels. It needs `stage_labels.csv` (18 kB) from the CAMELYON17 training folder.

In [ ]:
import urllib.request
if not os.path.exists(STAGE):
    try:
        urllib.request.urlretrieve("https://camelyon-dataset.s3.amazonaws.com/CAMELYON17/stages.csv", STAGE)
    except Exception as e:
        print("direct download failed:", e)
        !pip -q install awscli
        !aws s3 ls --no-sign-request s3://camelyon-dataset/CAMELYON17/training/
        # then: !aws s3 cp --no-sign-request s3://camelyon-dataset/<path shown above>/stage_labels.csv "$STAGE"
print(open(STAGE).read()[:200] if os.path.exists(STAGE) else f"Not found: put stage_labels.csv at {STAGE}")

In [ ]:
!python scripts/metastasis_size.py --root /content/data --stage_labels "$STAGE" --out "$SIZE"

## A. Hospital 1 experiments

### A0. Main ResNet-18 checkpoints (Person 1)
The final test needs the saved weights of the 20 main runs (4 methods × 5 seeds).

In [ ]:
n = len(glob.glob(f"{CKPT}/{MAIN_TAG}_*_seed*.pt"))
print(n, "main checkpoints found (need 20)")

In [ ]:
# ONLY if fewer than 20 were found: re-run the main study with checkpoints under a new tag, then use it.
# !python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds 0 1 2 3 4 --tag main --ckpt_dir "$CKPT"
# MAIN_TAG = "main"

### A1. HED strength (Person 2): σ = 0.05, 0.1 and 0.4, for HED and Both (σ = 0.2 is the main run; 0.4 tests whether a wider colour range helps)

In [ ]:
for s in ["0.05", "0.1", "0.4"]:
    !python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --methods hed both --sigma {s} --seeds {SEEDS} --tag abl_sigma{s}

### A2. Normalisation reference (Person 2): normalise to one lab's typical patch instead of the pooled one

In [ ]:
for r in ["h0", "h3", "h4"]:
    !python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --methods vahadane --ref {r} --seeds {SEEDS} --tag abl_ref_{r}

### A3. Stronger backbones (Person 1), same recipe, all four methods, with checkpoints
- `resnet18_histo`: the same ResNet-18, self-supervised on 57 histology datasets (Ciga et al. 2022).
- `resnet50_histo`: ResNet-50, MoCo v2 on ~33M TCGA + internal patches (Kang et al. 2023); no CAMELYON in its pretraining.

In [ ]:
import urllib.request, sys
sys.path.insert(0, "/content/ai-project")
from src.models import WEIGHT_URLS
W18 = f"{WEIGHTS}/tenpercent_resnet18.ckpt"; W50 = f"{WEIGHTS}/mocov2_rn50_ep200.torch"
for bb, path in (("resnet18_histo", W18), ("resnet50_histo", W50)):
    if not os.path.exists(path):
        urllib.request.urlretrieve(WEIGHT_URLS[bb], path)
    print(bb, os.path.getsize(path) // 2**20, "MB")

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds {SEEDS} --backbone resnet18_histo --weights_path "$W18" --tag bb_r18histo --ckpt_dir "$CKPT"

In [ ]:
!python scripts/run_experiments.py --root /content/data --cache "$CACHE" --out "$RESULTS" --seeds {SEEDS} --backbone resnet50_histo --weights_path "$W50" --tag bb_r50histo --ckpt_dir "$CKPT"

In [ ]:
# Hospital-1 summary so far (small text tables)
!python scripts/summarize.py --results "$RESULTS" --out "$SUMMARY"

## B. Final evaluation (Person 1, after A0 and A3 have finished)

B1 scores every patch of hospital 1 (34,904). **B2 is the one-time final test on hospital 2**: run it once, with every model we will report, and do not change any setting afterwards. Both write probabilities (for part D2) and a results table to `FINAL`.

In [ ]:
TAGS = f"{MAIN_TAG} bb_r18histo bb_r50histo"
!python scripts/predict_hospital.py --root /content/data --cache "$CACHE" --results "$RESULTS" --ckpt_dir "$CKPT" --tags {TAGS} --hospital 1 --out "$FINAL"

In [ ]:
# ONE TIME ONLY
!python scripts/predict_hospital.py --root /content/data --cache "$CACHE" --results "$RESULTS" --ckpt_dir "$CKPT" --tags {TAGS} --hospital 2 --out "$FINAL" --unseal_test

## C. Leave-one-hospital-out (Person 3)

Train on four hospitals, test on the fifth, for all five. Patch caches go to the local disk (about 3 GB per fold) and are deleted after each fold. About 60 ResNet-18 runs; resumable.

In [ ]:
for h in [0, 1, 3, 4, 2]:
    !python scripts/run_experiments.py --root /content/data --cache /content/cache_loho --out "$RESULTS" --split loho --test_hospital {h} --unseal_test --seeds {SEEDS} --tag loho_h{h}
    !rm -f /content/cache_loho/loho{h}_*

## D2. Metastasis size from each model's tumour calls (Person 1, after B)

In [ ]:
!python scripts/metastasis_size.py --root /content/data --stage_labels "$STAGE" --final "$FINAL" --hospitals 1 2 --out "$SIZE"

## E. Send the results back (everyone, at the end)

In [ ]:
!python scripts/summarize.py --results "$RESULTS" --out "$SUMMARY"
!cd "$DRIVE" && zip -q -r /content/send_back.zip results summary size final/*.csv
from google.colab import files
files.download("/content/send_back.zip")